In [2]:
import json
import pandas as pd
import numpy as np
import networkx as nx

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.sparse import hstack

import shap

/home/adithya/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [3]:
file_path = "../data/stix21/enterprise-attack.json"

with open(file_path) as f:
    stix = json.load(f)

if "spec_version" in stix:
    version = stix["spec_version"]

elif "objects" in stix and len(stix["objects"]) > 0:
    version = stix["objects"][0].get("spec_version", "2.0")

else:
    version = "2.0"

print("Detected Version:", version)
print("Total objects:", len(stix["objects"]))

Detected Version: 2.1
Total objects: 24772


In [4]:
records = []

for obj in stix["objects"]:
    if obj.get("type") == "relationship":
        continue

    record = {}
    record["id"] = obj.get("id")
    record["entity_type"] = obj.get("type")
    record["description"] = obj.get("description", "")
    record["confidence"] = obj.get("confidence", 50)
    record["external_ref_count"] = len(obj.get("external_references", []))
    record["validation_flag"] = 1 if obj.get("id") else 0

    records.append(record)

merged_df = pd.DataFrame(records)
merged_df = merged_df.dropna()
merged_df.head()

,id,entity_type,description,confidence,external_ref_count,validation_flag
0,x-mitre-collection--1f5f1533-f617-4ca8-9ab4-6a...,x-mitre-collection,ATT&CK for Enterprise provides a knowledge bas...,50,0,1
1,x-mitre-matrix--eafc1b4c-5e56-4965-bd4e-66a6a8...,x-mitre-matrix,Below are the tactics and technique representi...,50,1,1
2,course-of-action--00d7d21b-69d6-4797-88a2-c86f...,course-of-action,Ensure only valid password filters are registe...,50,2,1
3,course-of-action--02f0f92a-0a51-4c94-9bda-6437...,course-of-action,Prevent files from having a trailing space aft...,50,1,1
4,course-of-action--03c0c586-50ed-45a7-95f4-f496...,course-of-action,Prevent users from changing the <code>HISTCONT...,50,2,1


In [5]:
G = nx.DiGraph()

for obj in stix["objects"]:
    if obj.get("type") != "relationship":
        G.add_node(obj.get("id"))

for obj in stix["objects"]:
    if obj.get("type") == "relationship":
        G.add_edge(obj.get("source_ref"), obj.get("target_ref"))

degree = nx.degree_centrality(G)
pagerank = nx.pagerank(G)

merged_df["degree"] = merged_df["id"].map(lambda x: degree.get(x, 0))
merged_df["pagerank"] = merged_df["id"].map(lambda x: pagerank.get(x, 0))

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

Graph nodes: 4724
Graph edges: 20048


In [6]:
def threat_label(row):
    text = (str(row["entity_type"]) + " " + str(row["description"])).lower()
    keywords = ["malware", "attack", "ransom", "phish", "trojan"]
    for k in keywords:
        if k in text:
            return 1
    return 0

merged_df["Threat_Label"] = merged_df.apply(threat_label, axis=1)
merged_df["Threat_Label"].value_counts()

Threat_Label
0    2681
1    2043
Name: count, dtype: int64

In [7]:
merged_df["ml_text"] = (
    merged_df["entity_type"].astype(str) + " " +
    merged_df["description"].astype(str)
)

vectorizer = TfidfVectorizer(stop_words="english")
X_text = vectorizer.fit_transform(merged_df["ml_text"])

numeric_cols = ["confidence", "external_ref_count", "validation_flag", "degree", "pagerank"]

scaler = StandardScaler()
X_numeric = scaler.fit_transform(merged_df[numeric_cols])

X = hstack([X_text, X_numeric])
y = merged_df["Threat_Label"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "Logistic": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "MLP": MLPClassifier(max_iter=300)
}

scores = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:,1]
    roc = roc_auc_score(y_test, prob)
    scores[name] = roc
    print(name, "ROC:", round(roc,3))

best_model_name = max(scores, key=scores.get)
best_model = models[best_model_name]

print("Best Model:", best_model_name)

Logistic ROC: 0.994
RandomForest ROC: 0.999
MLP ROC: 0.991
Best Model: RandomForest


In [9]:
iso = IsolationForest(contamination=0.05, random_state=42)

merged_df["anomaly"] = iso.fit_predict(merged_df[numeric_cols])
merged_df["anomaly"] = merged_df["anomaly"].apply(lambda x: 1 if x == -1 else 0)

merged_df["anomaly"].value_counts()

anomaly
0    4487
1     237
Name: count, dtype: int64

In [10]:
merged_df["ml_probability"] = best_model.predict_proba(X)[:, 1]

def get_risk(row):
    ml   = float(row["ml_probability"])
    cred = float(row.get("credibility_score", 50)) / 100
    anom = int(row.get("anomaly_flag", 0))

    score = round((0.6 * ml + 0.2 * cred + 0.2 * (1 - anom)) * 100, 2)

    if score >= 85:
        return score, "CRITICAL"
    elif score >= 70:
        return score, "HIGH"
    elif score >= 40:
        return score, "MEDIUM"
    else:
        return score, "LOW"

merged_df["final_score"], merged_df["risk_level"] = zip(*merged_df.apply(get_risk, axis=1))

print(merged_df["risk_level"].value_counts())
merged_df[["id", "ml_probability", "final_score", "risk_level"]].head()

risk_level
LOW         2684
CRITICAL    1804
HIGH         211
MEDIUM        25
Name: count, dtype: int64


,id,ml_probability,final_score,risk_level
0,x-mitre-collection--1f5f1533-f617-4ca8-9ab4-6a...,0.05,4.0,LOW
1,x-mitre-matrix--eafc1b4c-5e56-4965-bd4e-66a6a8...,0.02,1.6,LOW
2,course-of-action--00d7d21b-69d6-4797-88a2-c86f...,0.09,7.2,LOW
3,course-of-action--02f0f92a-0a51-4c94-9bda-6437...,0.02,1.6,LOW
4,course-of-action--03c0c586-50ed-45a7-95f4-f496...,0.11,8.8,LOW


In [11]:
if best_model_name == "RandomForest":
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(
        X_test[:100].toarray(),
        check_additivity=False
    )
    print("SHAP explanation generated.")
else:
    print("Use RandomForest for SHAP explainability.")

SHAP explanation generated.


In [12]:
import pickle

pickle.dump(best_model, open("../models/model.pkl", "wb"))
pickle.dump(vectorizer, open("../models/vectorizer.pkl", "wb"))
pickle.dump(scaler, open("../models/scaler.pkl", "wb"))